In [1]:
#这两行代码是告诉ipynb，只要我运行cell，就去检查旁边.py文件看有没有被修改，如果修改了就重新读入

#读取之前用的函数
from functions import metrics_from_bt
#读取本地数据的代码
import pandas as pd
import numpy as np

# 1. 一步到位：读取 CSV，同时告诉 Pandas 把第 0 列当成索引，并尝试解析为时间格式
clean_data_path = "data/BTCUSDT_1h_6y_cleaned.csv"
df = pd.read_csv(clean_data_path, index_col=0, parse_dates=True)

# 2. (可选) 终极保险：强制确认一下索引是 UTC 时区格式
# 有时候 CSV 存取会丢失时区信息，加上这句可以保证万无一失
df.index = pd.to_datetime(df.index, utc=True)

# 检查结果
print("数据前 3 行：\n", df.head(3))
print("\n当前的索引类型：\n", type(df.index))

df6y=df

# 1) 全数据
df_full = df6y.copy()
df_full = df_full.sort_index()

# 2) 数据长度
n = len(df_full)

# 3) 切分位置
train_end = int(n * 0.6)
val_end   = int(n * 0.8)

# 4) 三段数据
train_df = df_full.iloc[:train_end].copy()
val_df   = df_full.iloc[train_end:val_end].copy()

print("Total rows :", len(df_full))
print("Train rows :", len(train_df))
print("Val rows   :", len(val_df))


数据前 3 行：
                               open     high      low    close       volume  \
2020-03-01 00:00:00+00:00  8523.61  8613.57  8511.11  8547.25  1952.740520   
2020-03-01 01:00:00+00:00  8546.65  8649.00  8514.06  8639.28  1901.273287   
2020-03-01 02:00:00+00:00  8640.23  8675.00  8617.73  8630.86  1271.182357   

                                                 close_time  \
2020-03-01 00:00:00+00:00  2020-03-01 00:59:59.999000+00:00   
2020-03-01 01:00:00+00:00  2020-03-01 01:59:59.999000+00:00   
2020-03-01 02:00:00+00:00  2020-03-01 02:59:59.999000+00:00   

                           quote_asset_volume  number_of_trades  \
2020-03-01 00:00:00+00:00        1.673789e+07           22876.0   
2020-03-01 01:00:00+00:00        1.630347e+07           24731.0   
2020-03-01 02:00:00+00:00        1.098729e+07           16257.0   

                           taker_buy_base_asset_volume  \
2020-03-01 00:00:00+00:00                   899.851144   
2020-03-01 01:00:00+00:00              

In [2]:

df = train_df.copy()

ret = df["close"].pct_change()
vol_window = 48

vol = ret.rolling(vol_window).std()
df_validation = val_df.copy()


print("Return / Vol ready")
from functions import backtest_multi_horizon_with_dispersion_consistency
from functions import backtest_multi_horizon_dynamic_threshold


# horizon（手动定义）
horizon_pairs = [
    (12,72),
    (24,168),
    (48,336)
]

print("Phase 6 setup ready")


# -------- volatility --------
vol_lookback = 72

# -------- risk targeting --------
target_vol_annual = 0.20
max_leverage = 3.0

# -------- dynamic threshold --------
threshold_window = 24 * 7
threshold_quantile = 0.7

# -------- transaction cost --------
fee_rate = 0.0004

print("Phase 7 parameters ready.")

Return / Vol ready
Phase 6 setup ready
Phase 7 parameters ready.


# Phase7  Robustness鲁棒性测试
## 查看对于基础参数进行小范围变动模型整体还稳定不稳定


## Cell 1 Baseline on validation V2模型的validation结果作为本节baseline 

In [3]:
# ============================================================
# Phase 7 — Baseline Model (Validation)
# ============================================================

bt_base_val = backtest_multi_horizon_with_dispersion_consistency(
    df=df_validation,
    horizon_pairs=horizon_pairs,
    vol_lookback=vol_lookback,
    target_vol_annual=target_vol_annual,
    threshold_window=threshold_window,
    threshold_quantile=threshold_quantile,
    fee_rate=fee_rate,
    max_leverage=max_leverage,
)

baseline_metrics_val = metrics_from_bt(bt_base_val)

print("Baseline (Validation):")
print(baseline_metrics_val)

Baseline (Validation):
(2.0208534365771467, -0.07472342857384173, 0.2815964176309842)


## Cell 2 对交易费用进行变动查看运行结果如何

In [4]:
# ============================================================
# Phase 7 — Cost Stress Test
# ============================================================

fee_list = [0.0004, 0.0008, 0.0012]

results_cost = []

for fee in fee_list:
    bt = backtest_multi_horizon_with_dispersion_consistency(
        df=df_validation,
        horizon_pairs=horizon_pairs,
        vol_lookback=vol_lookback,
        target_vol_annual=target_vol_annual,
        threshold_window=threshold_window,
        threshold_quantile=threshold_quantile,
        fee_rate=fee,
        max_leverage=max_leverage,
    )

    sharpe, maxdd, totalret = metrics_from_bt(bt)
    results_cost.append([fee, sharpe, maxdd, totalret])

results_cost_df = pd.DataFrame(
    results_cost,
    columns=["fee_rate", "sharpe", "maxdd", "totalret"]
)

results_cost_df

,fee_rate,sharpe,maxdd,totalret
0,0.0004,2.020853,-0.074723,0.281596
1,0.0008,1.668406,-0.092530,0.226047
2,0.0012,1.315961,-0.115064,0.172900


## Cell3 Threshold Robustness

In [5]:
# ============================================================
# Phase 7 — Threshold Quantile Robustness
# ============================================================

quantile_list = [0.6, 0.7, 0.8]

results_q = []

for q in quantile_list:
    bt = backtest_multi_horizon_with_dispersion_consistency(
        df=df_validation,
        horizon_pairs=horizon_pairs,
        vol_lookback=vol_lookback,
        target_vol_annual=target_vol_annual,
        threshold_window=threshold_window,
        threshold_quantile=q,
        fee_rate=fee_rate,
        max_leverage=max_leverage,
    )

    sharpe, maxdd, totalret = metrics_from_bt(bt)
    results_q.append([q, sharpe, maxdd, totalret])

results_q_df = pd.DataFrame(
    results_q,
    columns=["threshold_quantile", "sharpe", "maxdd", "totalret"]
)

results_q_df

,threshold_quantile,sharpe,maxdd,totalret
0,0.6,1.931728,-0.088474,0.294706
1,0.7,2.020853,-0.074723,0.281596
2,0.8,2.307140,-0.072248,0.280724


## Cell 4 改变horizon 参数查看模型情况

In [6]:
# ============================================================
# Phase 7 — Horizon Perturbation
# ============================================================

horizon_variants = {
    "base": [(12, 72), (24, 168), (48, 336)],
    "shift_1": [(12, 60), (24, 144), (48, 300)],
    "shift_2": [(16, 72), (32, 168), (64, 336)],
}

results_h = []

for name, h_pairs in horizon_variants.items():
    bt = backtest_multi_horizon_with_dispersion_consistency(
        df=df_validation,
        horizon_pairs=h_pairs,
        vol_lookback=vol_lookback,
        target_vol_annual=target_vol_annual,
        threshold_window=threshold_window,
        threshold_quantile=threshold_quantile,
        fee_rate=fee_rate,
        max_leverage=max_leverage,
    )

    sharpe, maxdd, totalret = metrics_from_bt(bt)
    results_h.append([name, sharpe, maxdd, totalret])

results_h_df = pd.DataFrame(
    results_h,
    columns=["variant", "sharpe", "maxdd", "totalret"]
)

results_h_df

,variant,sharpe,maxdd,totalret
0,base,2.020853,-0.074723,0.281596
1,shift_1,1.946667,-0.077926,0.263911
2,shift_2,1.924503,-0.080520,0.258965


This model demonstrates strong robustness across:

- transaction cost stress
- parameter perturbation 参数扰动
- structural variation 结构变化

Performance degradation is smooth and controlled,
indicating that the strategy is not overfit and has genuine signal stability.

# V2总结

## 为什么不继续调整模型？
在进行到现在阶段，我们使用了validation上叠filter，或许可以在vol上再加一个validation，在vol上+cooldown或者其他改进。但是V2就完结不了，可以在后续V3研究路线上研究模型改进。

## V2的核心结构和参数：

## Strategy V2 — Final Candidate Model (Before Final Test)

| Component | Setting |
|---|---|
| Strategy Name | V2 |
| Signal Type | EMA trend |
| Horizon Structure | Multi-horizon Reduced A |
| Horizon Pairs | (12,72), (24,168), (48,336) |
| Signal Formula | trend = EMA_fast / EMA_slow - 1 |
| Vol Normalization | signal = trend / realized_vol |
| Volatility Lookback | 72 |
| Dynamic Threshold | Rolling quantile threshold |
| Threshold Window | 24 * 7 = 168 hours |
| Threshold Quantile (baseline) | 0.7 |
| Threshold Quantile Robustness Range | 0.6, 0.7, 0.8 |
| Signal Mapping | tanh(filtered_signal) |
| Alpha Combination | Equal-weight mean across horizon directions |
| Regime Filter | Dispersion-based continuous consistency |
| Dispersion Definition | Cross-horizon std of directions |
| Consistency Definition | consistency = clip(1 - dispersion, 0, 1) |
| Continuous Scaling | scaled_direction = combined_direction * consistency |
| Target Annual Volatility | 0.20 |
| Target Hourly Volatility | 0.20 / sqrt(24*365) |
| Max Leverage | 3.0 |
| Transaction Cost Baseline | 0.0004 |
| Cost Stress Range | 0.0004, 0.0008, 0.0012 |
| Validation Baseline Sharpe | 2.021 |
| Validation Baseline Max Drawdown | -0.075 |
| Validation Baseline Total Return | 0.282 |

## Model Freeze Decision

| Question | Answer |
|---|---|
| Is V2 the current best model? | Yes |
| Can V2 still be improved? | Probably yes |
| Should we improve it now? | No |
| Why not? | Because validation-based model selection is complete |
| What is the correct next step? | Freeze V2 and run final one-time test |
| When can new ideas (e.g. vol + trend filter) be tried? | After final test, as V3 research |